# Pipeline de datos: Mercadona + Open Food Facts (Bronze → Silver → Gold)

Este notebook contiene todo el proceso de preparación de datos de mi TFM. El proyecto es una herramienta que ayuda a hacer la compra en Mercadona según un objetivo nutricional (por ejemplo, comer más proteína o menos calorías). Para eso necesito una tabla con productos reales de Mercadona, con su precio y sus valores nutricionales.

**De dónde salen los datos**
- Mercadona (https://tienda.mercadona.es): nombre, precio, tamaño del envase y código de barras (EAN) de cada producto. Mercadona no ofrece una API pública, así que uso la misma que utiliza su tienda online. Lo hago solo con fines académicos.
- Open Food Facts (https://world.openfoodfacts.org): los valores nutricionales por cada 100 g. Mercadona no los publica en un formato que se pueda leer automáticamente, así que los busco aquí a partir del código de barras. Estos datos contienen información de Open Food Facts, disponible bajo la licencia Open Database License (ODbL, https://opendatacommons.org/licenses/odbl/1-0/).

**Qué productos incluye esta versión:** arroces, legumbres, y pasta y fideos.

**Cómo está organizado el proceso.** Los datos pasan por tres fases:
- Bronze: los datos tal cual los devuelven las páginas, sin tocarlos.
- Silver: los datos ya limpios, unidos entre sí y con marcas que indican su calidad.
- Gold: la tabla final, con una fila por producto y solo los datos que se pueden usar.

**Cómo ejecutarlo:** de arriba abajo. Lo que ya se haya descargado no se vuelve a pedir.

In [1]:
import json
import os
import time
import shutil
import datetime

import numpy as np
import pandas as pd
import requests

# OFF pide identificarse. Cambia el texto por tu nombre o email.
headers = {"User-Agent": "TFM-DataScience/0.1 (Aroa)"}

# Carpetas de datos (se crean si no existen)
for carpeta in [
    "data/raw/mercadona/categories",
    "data/raw/mercadona/products",
    "data/raw/openfoodfacts",
    "data/silver",
    "data/gold",
]:
    os.makedirs(carpeta, exist_ok=True)

## 1. Bronze: descarga de datos en crudo

Por cada categoría de Mercadona hago tres cosas: descargo la lista de productos, descargo la ficha de cada producto (que es donde aparece su código de barras) y, con ese código, busco el producto en Open Food Facts. Todo se guarda sin modificar en la carpeta data/raw.

Tres decisiones que he tomado:
- Uno los datos de las dos fuentes por el código de barras y nunca por el nombre. Los nombres cambian de una web a otra y pueden confundirse; el código de barras es único para cada producto.
- Si un archivo ya está descargado, lo reutilizo. Así no repito peticiones y respeto a las webs.
- Open Food Facts bloquea durante un rato a quien le hace demasiadas peticiones seguidas. Cuando pasa, el código espera unos segundos y lo vuelve a intentar. Si aun así falla, el producto queda marcado en la columna off_error.

In [2]:
def pedir_off(ean, headers, intentos=3):
    url = f"https://world.openfoodfacts.org/api/v2/product/{ean}.json"
    for intento in range(intentos):
        r = requests.get(url, headers=headers)
        if r.status_code in (200, 404):
            try:
                return r.json()
            except ValueError:
                pass
        print("Open Food Facts respondió", r.status_code, "para", ean, "- reintento")
        time.sleep(5 * (intento + 1))
    return None


def bronze_categoria(category_id, headers):
    # 1) Categoría
    url = f"https://tienda.mercadona.es/api/categories/{category_id}/?lang=es&wh=vlc1"
    r = requests.get(url)
    r.raise_for_status()
    datos = r.json()
    with open(f"data/raw/mercadona/categories/{category_id}.json", "w", encoding="utf-8") as f:
        json.dump(datos, f, ensure_ascii=False, indent=2)

    productos = []
    for grupo in datos["categories"]:
        productos.extend(grupo["products"])

    resultados = []
    for p in productos:
        product_id = p["id"]

        # 2) Detalle en Mercadona (si ya está descargado, se reutiliza)
        ruta = f"data/raw/mercadona/products/{product_id}.json"
        if os.path.exists(ruta):
            with open(ruta, encoding="utf-8") as f:
                detalle = json.load(f)
        else:
            r = requests.get(f"https://tienda.mercadona.es/api/products/{product_id}/?lang=es&wh=vlc1")
            if r.status_code != 200:
                print("Error en Mercadona:", product_id, r.status_code)
                continue
            detalle = r.json()
            with open(ruta, "w", encoding="utf-8") as f:
                json.dump(detalle, f, ensure_ascii=False, indent=2)
            time.sleep(1)

        # 3) Ficha en Open Food Facts (si ya está descargada, se reutiliza)
        ean = detalle.get("ean")
        encontrado = False
        off_error = False
        if ean:
            ruta_off = f"data/raw/openfoodfacts/{ean}.json"
            if os.path.exists(ruta_off):
                with open(ruta_off, encoding="utf-8") as f:
                    off = json.load(f)
            else:
                off = pedir_off(ean, headers)
                if off is not None:
                    with open(ruta_off, "w", encoding="utf-8") as f:
                        json.dump(off, f, ensure_ascii=False, indent=2)
                time.sleep(1)
            if off is None:
                off_error = True
            else:
                encontrado = off.get("status") == 1

        resultados.append({
            "categoria_id": category_id,
            "product_id": product_id,
            "nombre": detalle["display_name"],
            "ean": ean,
            "off_encontrado": encontrado,
            "off_error": off_error,
        })

    return pd.DataFrame(resultados)

In [3]:
df_118 = bronze_categoria(118, headers)   # Arroces
df_121 = bronze_categoria(121, headers)   # Legumbres
df_120 = bronze_categoria(120, headers)   # Pasta y fideos

bronze = pd.concat([df_118, df_121, df_120], ignore_index=True)
print("Productos descargados:", len(bronze))

Productos descargados: 88


### ¿Cuántos productos encuentro en Open Food Facts?

Antes de seguir quiero saber cuántos productos de Mercadona existen también en Open Food Facts, porque de eso depende cuántos tendrán datos nutricionales. En mi descarga, 77 de los 88 productos (el 87,5 %) aparecen. Los 11 que no aparecen son todos de la marca Hacendado, que además es el 80 % de los productos descargados. No sé por qué pasa, así que lo dejo anotado como algo que he observado y no como una explicación.

También hay un código de barras con 12 dígitos en lugar de 13 que lo marco para tenerlo localizado.

In [4]:
cobertura = bronze.groupby("categoria_id").agg(
    productos=("product_id", "count"),
    encontrados=("off_encontrado", "sum"),
    errores_off=("off_error", "sum"),
)
cobertura["pct_encontrados"] = (cobertura["encontrados"] / cobertura["productos"] * 100).round(1)
print(cobertura)

# ¿Los que faltan son de una marca concreta?
bronze["es_hacendado"] = bronze["nombre"].str.contains("Hacendado", case=False)
print(bronze.groupby("es_hacendado")["off_encontrado"].agg(["sum", "count"]))

# Productos que no están en OFF
print(bronze[~bronze["off_encontrado"]][["categoria_id", "nombre", "ean"]])

              productos  encontrados  errores_off  pct_encontrados
categoria_id                                                      
118                  14           13            0             92.9
120                  56           49            0             87.5
121                  18           15            0             83.3
              sum  count
es_hacendado            
False          18     18
True           59     70
    categoria_id                                            nombre  \
1            118                             Arroz largo Hacendado   
21           121                      Alubia cocida roja Hacendado   
22           121                     Alubia cocida pinta Hacendado   
26           121                    Alubia cocida granja Hacendado   
45           120                                Macarrón Hacendado   
51           120                                Macarrón Hacendado   
59           120                          Spaghetti fino Hacendado   
62   

## 2. Silver: limpiar y unir los datos

En esta fase preparo los datos para poder usarlos:
1. Elimino los productos repetidos (un mismo producto puede aparecer en dos categorías).
2. Saco de las fichas de Open Food Facts los valores nutricionales por cada 100 g: calorías, proteína, grasa, hidratos, y fibra, azúcares y sal cuando están.
3. Marco qué productos tienen completos los cuatro datos principales (calorías, proteína, grasa e hidratos).

No borro los productos que tienen datos incompletos. Los dejo marcados para poder contar después cuántos se pierden y por qué.

In [5]:
antes = len(bronze)
silver = bronze.drop_duplicates(subset="product_id").copy()
print("Filas antes:", antes, "| después:", len(silver))

campos = ["energy-kcal_100g", "proteins_100g", "fat_100g", "carbohydrates_100g"]
extras = ["fiber_100g", "sugars_100g", "salt_100g"]

filas = []
for _, fila in silver[silver["off_encontrado"]].iterrows():
    with open(f"data/raw/openfoodfacts/{fila['ean']}.json", encoding="utf-8") as f:
        off = json.load(f)
    n = off["product"].get("nutriments", {})
    registro = {"product_id": fila["product_id"]}
    for c in campos + extras:
        registro[c] = n.get(c)
    filas.append(registro)

silver = silver.merge(pd.DataFrame(filas), on="product_id", how="left")

silver["macros_completos"] = silver[campos].notna().all(axis=1)
silver["ean_estandar"] = silver["ean"].astype(str).str.len() == 13

silver["macros_completos"].value_counts()

Filas antes: 88 | después: 88


macros_completos
True     72
False    16
Name: count, dtype: int64

### ¿Son fiables los valores nutricionales?

Open Food Facts es una base de datos que rellena cualquier persona, no una fuente oficial. Eso la hace muy útil, porque tiene muchísimos productos, pero también significa que alguna ficha puede tener un dato mal introducido. Por eso compruebo los datos antes de usarlos.

La comprobación es esta: las calorías de un alimento se pueden calcular a partir de sus nutrientes (4 calorías por gramo de proteína, 4 por gramo de hidratos, 9 por gramo de grasa y 2 por gramo de fibra). Comparo ese cálculo con las calorías que declara la ficha. Si se parecen, el dato es coherente.

Al principio usé una regla sencilla (diferencia máxima del 10 %), pero daba muchas falsas alarmas. El motivo es que muchas fichas no incluyen la fibra, y la fibra solo puede sumar calorías, nunca restarlas. Por eso clasifico cada producto en cuatro grupos:
- coherente: el cálculo y las calorías declaradas coinciden (diferencia de hasta el 10 %).
- no_concluyente: no hay dato de fibra y las calorías declaradas son entre un 10 % y un 25 % más altas que el cálculo. Puede deberse a la fibra que falta, así que no puedo asegurar que haya un error.
- incoherente: la diferencia es demasiado grande para explicarla con la fibra.
- sin_datos: faltan valores nutricionales.

Para saber si la comprobación acierta, miré a mano las etiquetas de dos productos marcados como incoherentes (el arroz cocido redondo Sabroz y la quinoa cocida Sabroz). En los dos casos el dato de Open Food Facts estaba mal.

In [7]:
silver["kcal_calculadas"] = (
    4 * silver["proteins_100g"] + 4 * silver["carbohydrates_100g"] + 9 * silver["fat_100g"]
)
silver["diferencia_pct"] = (
    (silver["energy-kcal_100g"] - silver["kcal_calculadas"]) / silver["kcal_calculadas"] * 100
).round(1)


def estado_coherencia(fila):
    if not fila["macros_completos"]:
        return "sin_datos"
    calc = fila["kcal_calculadas"]
    kcal = fila["energy-kcal_100g"]
    fibra = fila["fiber_100g"]

    # Con fibra conocida (mayor que 0): sumamos 2 kcal por gramo
    if pd.notna(fibra) and fibra > 0:
        esperado = calc + 2 * fibra
        dif = (kcal - esperado) / esperado * 100
        return "coherente" if abs(dif) <= 10 else "incoherente"

    # Sin fibra: la fibra solo puede añadir kcal, nunca quitarlas
    dif = (kcal - calc) / calc * 100
    if abs(dif) <= 10:
        return "coherente"
    if 10 < dif <= 25:
        return "no_concluyente"
    return "incoherente"


silver["estado_calidad"] = silver.apply(estado_coherencia, axis=1)
silver["estado_calidad"].value_counts()

estado_calidad
coherente         64
sin_datos         16
no_concluyente     6
incoherente        2
Name: count, dtype: int64

### Precio, tamaño y formato

El precio, el tamaño del envase y el formato (paquete, pack, etc.) vienen de la ficha del producto en Mercadona. Mercadona da un precio de referencia por kilo o por litro; en esta descarga los 88 productos lo tienen en euros por kilo, así que no hace falta convertir unidades. En algunos productos el formato no viene indicado, y lo dejo vacío en lugar de inventarlo.

In [8]:
filas_precio = []
for pid in silver["product_id"]:
    with open(f"data/raw/mercadona/products/{pid}.json", encoding="utf-8") as f:
        d = json.load(f)
    pi = d["price_instructions"]
    filas_precio.append({
        "product_id": pid,
        "precio_unidad": float(pi["unit_price"]),
        "precio_referencia": float(pi["reference_price"]),
        "unidad_referencia": pi["reference_format"],
        "tamano_envase": pi["unit_size"],
        "unidad_tamano": pi["size_format"],
        "formato": d.get("packaging"),
    })

silver = silver.merge(pd.DataFrame(filas_precio), on="product_id", how="left")
print(silver["unidad_referencia"].value_counts())

unidad_referencia
kg    88
Name: count, dtype: int64


### ¿Está el producto seco, cocinado o fresco?

Esto importa porque no se pueden comparar bien un arroz seco y un arroz ya cocinado. El cocinado lleva agua, así que tiene menos calorías y menos proteína en cada 100 g, y además cuesta más por kilo. Si los mezclo, una cuenta como "proteína por euro" daría resultados engañosos.

Averiguo el estado a partir del nombre del producto:
- seco: es el valor por defecto.
- cocinado: el nombre contiene "cocid" pero no "precocid", porque las placas de lasaña precocidas se venden secas.
- fresco: el nombre contiene "fresca" (pasta fresca, gnocchi...).
- instantaneo: fideos orientales y ramen. Tienen entre 93 y 204 calorías por 100 g, que son muy pocas para unos fideos instantáneos secos. Lo más probable es que los datos se refieran al plato ya preparado, aunque no lo he podido confirmar. Por eso este grupo queda aparte y no se compara con los demás.

Para comprobar que el reparto tiene sentido miro las calorías de cada grupo: los cocinados quedan por debajo de unas 165 calorías, los frescos entre 214 y 306, y los secos por encima de 290. Clasificar por el nombre es un método sencillo y puede fallar con productos nuevos.

In [9]:
nombre = silver["nombre"].str.lower()
condiciones = [
    nombre.str.contains("orientales|ramen"),
    nombre.str.contains("cocid") & ~nombre.str.contains("precocid"),
    nombre.str.contains("fresca"),
]
silver["estado"] = np.select(condiciones, ["instantaneo", "cocinado", "fresco"], default="seco")

# Los rangos de kcal por estado deben ser coherentes (solo productos con kcal)
print(silver.groupby("estado")["energy-kcal_100g"].agg(["count", "min", "max"]))

silver.to_csv("data/silver/productos_silver.csv", index=False)
silver.shape

             count    min    max
estado                          
cocinado        11   83.0  164.0
fresco           9  214.0  306.0
instantaneo      4   93.0  204.0
seco            48  292.0  394.0


(88, 26)

## 3. Gold: la tabla final

La tabla final tiene una fila por cada producto de Mercadona. Entran los productos cuyos datos nutricionales han pasado la comprobación: los marcados como coherente (64) y los no_concluyente (6). La columna estado_calidad se queda en la tabla para que, quien la use, pueda decidir si quiere ser más estricto y quedarse solo con los coherentes. Los productos incoherentes o sin datos (18) no entran, pero siguen en Silver para poder saber qué se descartó.

Añado tres columnas calculadas, pensadas para el objetivo del proyecto:
- proteina_g_por_euro: cuántos gramos de proteína se obtienen por cada euro.
- kcal_por_euro: cuántas calorías se obtienen por cada euro.
- pct_kcal_proteina: qué porcentaje de las calorías del producto viene de la proteína.

Estas columnas solo se pueden comparar entre productos que estén en el mismo estado (seco con seco, cocinado con cocinado).

In [10]:
gold = silver[silver["estado_calidad"].isin(["coherente", "no_concluyente"])].copy()

gold = gold.rename(columns={
    "energy-kcal_100g": "kcal_100g",
    "proteins_100g": "proteina_100g",
    "fat_100g": "grasa_100g",
    "carbohydrates_100g": "hidratos_100g",
    "fiber_100g": "fibra_100g",
    "sugars_100g": "azucares_100g",
    "salt_100g": "sal_100g",
})

gold["proteina_g_por_euro"] = (gold["proteina_100g"] * 10 / gold["precio_referencia"]).round(1)
gold["kcal_por_euro"] = (gold["kcal_100g"] * 10 / gold["precio_referencia"]).round(0)
gold["pct_kcal_proteina"] = (gold["proteina_100g"] * 4 / gold["kcal_100g"] * 100).round(1)

columnas = [
    "product_id", "ean", "nombre", "categoria_id", "formato", "estado",
    "precio_unidad", "precio_referencia", "tamano_envase",
    "kcal_100g", "proteina_100g", "grasa_100g", "hidratos_100g",
    "fibra_100g", "azucares_100g", "sal_100g",
    "proteina_g_por_euro", "kcal_por_euro", "pct_kcal_proteina",
    "estado_calidad",
]
gold = gold[columnas]

gold.to_csv("data/gold/productos_gold.csv", index=False)
print(gold.shape)
print(gold["estado"].value_counts())
gold.sort_values("proteina_g_por_euro", ascending=False).head(10)

(70, 20)
estado
seco           48
cocinado        9
fresco          9
instantaneo     4
Name: count, dtype: int64


,product_id,ean,nombre,categoria_id,formato,estado,precio_unidad,precio_referencia,tamano_envase,kcal_100g,proteina_100g,grasa_100g,hidratos_100g,fibra_100g,azucares_100g,sal_100g,proteina_g_por_euro,kcal_por_euro,pct_kcal_proteina,estado_calidad
28,5330,8480000053305,Lenteja pardina Hacendado,121,Paquete,seco,1.85,1.85,1.0,320.0,24.5,1.2,48.1,9.6,1.3,0.05,132.4,1730.0,30.6,coherente
23,5124,8480000051240,Alubia blanca Hacendado,121,Paquete,seco,2.45,2.45,1.0,292.0,21.4,1.0,40.0,18.9,2.3,0.03,87.3,1192.0,29.3,coherente
17,5214,8480000052148,Garbanzo Hacendado,121,Paquete,seco,2.15,2.15,1.0,348.0,18.6,6.1,47.5,14.4,6.0,0.10,86.5,1619.0,21.4,coherente
24,67608,8480000676085,Alubia negra Hacendado,121,Paquete,seco,2.50,2.50,1.0,292.0,21.4,1.0,40.0,18.9,2.3,0.03,85.6,1168.0,29.3,coherente
27,23453,8480000234537,Alubia pinta Hacendado,121,Paquete,seco,2.30,2.30,1.0,349.0,19.0,1.4,52.5,25.4,2.2,0.03,82.6,1517.0,21.8,coherente
42,6344,8402001015717,Hélices con vegetales Hacendado,120,Paquete,seco,1.60,1.60,1.0,362.0,13.0,1.5,76.0,4.0,0.0,0.00,81.2,2262.0,14.4,coherente
43,6260,8480000062604,Macarrón fino Hacendado,120,Paquete,seco,0.85,1.70,0.5,361.0,13.0,1.5,72.0,3.5,3.6,0.01,76.5,2124.0,14.4,coherente
49,6244,8480000062444,Pasta tiburón Hacendado,120,Paquete,seco,0.85,1.70,0.5,361.0,13.0,1.5,72.0,3.5,3.6,0.01,76.5,2124.0,14.4,coherente
36,6258,8480000062581,Pasta maravilla Hacendado,120,Paquete,seco,0.85,1.70,0.5,361.0,13.0,1.5,72.0,3.5,3.6,0.01,76.5,2124.0,14.4,coherente
35,6363,8480000063632,Pasta estrellas Hacendado,120,Paquete,seco,0.85,1.70,0.5,361.0,13.0,1.5,72.0,3.5,3.6,0.01,76.5,2124.0,14.4,coherente


## 4. Muestra de datos para el repositorio

Los datos descargados completos no se suben a GitHub, porque se pueden volver a obtener ejecutando este notebook. En su lugar guardo una muestra pequeña: tres productos tal cual se descargaron, las diez primeras filas de la tabla final y un archivo con la fecha de descarga.

La fecha es importante porque el catálogo de Mercadona cambia con el tiempo: si alguien repite el proceso más adelante, puede encontrar algunos productos distintos.

In [11]:
muestra = bronze[bronze["off_encontrado"]].drop_duplicates("product_id").head(3)

os.makedirs("data/sample/raw/mercadona/products", exist_ok=True)
os.makedirs("data/sample/raw/openfoodfacts", exist_ok=True)

for _, fila in muestra.iterrows():
    shutil.copy(f"data/raw/mercadona/products/{fila['product_id']}.json",
                "data/sample/raw/mercadona/products/")
    shutil.copy(f"data/raw/openfoodfacts/{fila['ean']}.json",
                "data/sample/raw/openfoodfacts/")

gold.head(10).to_csv("data/sample/gold_muestra.csv", index=False)

with open("data/sample/fecha_descarga.txt", "w") as f:
    f.write(f"Datos descargados el {datetime.date.today()}\n")

with open(".gitignore", "w") as f:
    f.write("data/raw/\n.ipynb_checkpoints/\n__pycache__/\n.DS_Store\n")